# 📊 Lesson 3.6 — Model Evaluation: Measuring What Actually Matters

**AI Crash Course for Absolute Beginners**

Accuracy alone is misleading. In this notebook:
- Build every key classification metric from scratch
- Interpret confusion matrices
- Plot ROC and Precision-Recall curves
- Calculate regression metrics: MAE, RMSE, R²
- Do hyperparameter tuning with GridSearchCV

> Run each cell with **Shift + Enter**.

In [ ]:
!pip install numpy pandas matplotlib scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_breast_cancer   # datasets for testing our metrics
from sklearn.model_selection import train_test_split, GridSearchCV
# GridSearchCV — automatically tries every combination of hyperparameters and picks the best

from sklearn.ensemble import RandomForestClassifier   # ensemble of many decision trees
from sklearn.linear_model import LogisticRegression   # classic baseline classifier
from sklearn.metrics import (
    confusion_matrix,            # table: predicted vs actual class counts
    ConfusionMatrixDisplay,      # helper to draw the confusion matrix as a heatmap
    roc_curve,                   # computes the ROC curve (FPR vs TPR at every threshold)
    auc,                         # calculates the area under a curve (higher = better)
    precision_recall_curve,      # computes precision and recall at every threshold
    average_precision_score,     # single number summarising the precision-recall curve
    classification_report,       # prints precision, recall, F1 for every class
    accuracy_score, precision_score, recall_score, f1_score   # individual metric functions
)

---
## Part 1 — Classification Metrics from First Principles

The four cells of a confusion matrix:

In [ ]:
# Simulate a spam detector
y_true = [1,1,1,1,1,0,0,0,0,0,1,1,0,0,1]  # 1=spam
y_pred = [1,1,1,0,1,0,0,1,0,0,1,0,0,0,1]

# zip(y_true, y_pred) pairs each true label with the predicted label
# We count how often each combination occurs:
#   TP = True Positive  — model said spam AND it was spam
#   TN = True Negative  — model said not spam AND it wasn't spam
#   FP = False Positive — model said spam BUT it was not spam (wrong alarm)
#   FN = False Negative — model said not spam BUT it was spam (missed it)
TP = sum(1 for t,p in zip(y_true,y_pred) if t==1 and p==1)  # correctly flagged spam
TN = sum(1 for t,p in zip(y_true,y_pred) if t==0 and p==0)  # correctly passed ham
FP = sum(1 for t,p in zip(y_true,y_pred) if t==0 and p==1)  # ham flagged as spam
FN = sum(1 for t,p in zip(y_true,y_pred) if t==1 and p==0)  # spam missed

accuracy  = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)    # of all flagged, how many were actually spam?
recall    = TP / (TP + FN)    # of all spam, how many did we catch?
f1        = 2 * precision * recall / (precision + recall)   # harmonic mean of precision and recall

print(f"TP={TP}  TN={TN}  FP={FP}  FN={FN}")
print(f"\nAccuracy  : {accuracy:.2f}  (correct predictions / total)")
print(f"Precision : {precision:.2f}  (quality of positive predictions)")
print(f"Recall    : {recall:.2f}  (coverage of actual positives)")
print(f"F1 Score  : {f1:.2f}  (harmonic mean of precision + recall)")

In [ ]:
# The accuracy paradox: imbalanced classes
# Imagine a dataset where 95% of patients are healthy and only 5% have a rare disease
y_true_imb = [0]*950 + [1]*50          # 950 healthy (class 0), 50 sick (class 1)
y_pred_dumb = [0]*1000                  # a "lazy" model that ALWAYS predicts healthy

# This lazy model never learns anything — but watch what accuracy says:
acc_dumb = accuracy_score(y_true_imb, y_pred_dumb)

# recall_score measures what fraction of actual positives (sick patients) were caught
# zero_division=0 prevents an error when the model never predicts class 1 (TP+FN=0)
rec_dumb = recall_score(y_true_imb, y_pred_dumb, zero_division=0)

print(f"Dumb model accuracy : {acc_dumb:.0%}  <- looks great!")
print(f"Dumb model recall   : {rec_dumb:.0%}  <- missed 100% of the rare class!")
print("\nLesson: Always check Recall and F1, not just accuracy.")

---
## Part 2 — Confusion Matrix on Real Data

In [ ]:
data = load_breast_cancer()
X_tr, X_te, y_tr, y_te = train_test_split(data.data, data.target,
                                           test_size=0.25, stratify=data.target,
                                           random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_te)

cm = confusion_matrix(y_te, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(ax=ax, colorbar=False,
    cmap="Blues")
ax.set_title("Random Forest — Breast Cancer")
plt.tight_layout()
plt.show()

print(classification_report(y_te, y_pred, target_names=data.target_names))

---
## Part 3 — ROC Curve and AUC

In [ ]:
# Get probability scores (not just the 0/1 prediction, but how confident the model is)
# predict_proba returns two columns: [[prob class 0, prob class 1], ...]
# [:, 1] selects the second column — the probability that the sample is class 1 (malignant)
y_proba_rf = rf.predict_proba(X_te)[:, 1]

lr = LogisticRegression(max_iter=5000, random_state=42).fit(X_tr, y_tr)
y_proba_lr = lr.predict_proba(X_te)[:, 1]

fig, ax = plt.subplots(figsize=(6, 5))

for model_name, proba in [("Random Forest", y_proba_rf), ("Logistic Regression", y_proba_lr)]:
    # roc_curve computes the False Positive Rate and True Positive Rate at every possible
    # probability threshold (from 0 to 1) — this gives you the full curve, not just one point
    fpr, tpr, _ = roc_curve(y_te, proba)
    # auc (Area Under Curve) summarises the whole ROC curve as one number
    # 1.0 = perfect classifier | 0.5 = no better than flipping a coin
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {roc_auc:.3f})")

ax.plot([0,1],[0,1],"k--", linewidth=1, label="Random classifier (AUC=0.5)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("ROC Curves")
ax.legend()
plt.tight_layout()
plt.show()
print("AUC = 1.0 is perfect. AUC = 0.5 is no better than random guessing.")

---
## Part 4 — Precision-Recall Curve (Better for Imbalanced Data)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for model_name, proba in [("Random Forest", y_proba_rf), ("Logistic Regression", y_proba_lr)]:
    prec, rec, _ = precision_recall_curve(y_te, proba)
    ap = average_precision_score(y_te, proba)
    ax.plot(rec, prec, linewidth=2, label=f"{model_name} (AP = {ap:.3f})")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves")
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 5 — Hyperparameter Tuning with GridSearchCV

In [ ]:
# param_grid defines all the hyperparameter values to try
# GridSearchCV will test EVERY combination — 3 x 3 x 2 = 18 combinations total
param_grid = {
    "n_estimators" : [50, 100, 200],       # how many trees in the forest
    "max_depth"    : [None, 10, 20],        # None = unlimited depth
    "min_samples_split": [2, 5]             # minimum samples needed to split a node
}

# GridSearchCV wraps the model and param grid:
# cv=5 means evaluate each combination with 5-fold cross-validation
# scoring="f1" means optimise for F1 score (good for imbalanced data)
# n_jobs=-1 uses all CPU cores to run searches in parallel (much faster)
# verbose=1 prints progress updates while searching
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5, scoring="f1", n_jobs=-1, verbose=1
)
grid_search.fit(X_tr, y_tr)

# best_params_ is a dictionary of the hyperparameter values that scored highest
print("Best parameters:", grid_search.best_params_)
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")

# best_estimator_ is the actual trained model using those best parameters
best_model = grid_search.best_estimator_
y_best = best_model.predict(X_te)
print(f"Test F1 with best model: {f1_score(y_te, y_best):.4f}")

---
## Quick Reference Card

| Metric | Formula | Use when |
|---|---|---|
| Accuracy | (TP+TN)/total | Balanced classes |
| Precision | TP/(TP+FP) | False positives are costly |
| Recall | TP/(TP+FN) | False negatives are costly (medical) |
| F1 Score | 2·P·R/(P+R) | Imbalanced classes |
| AUC-ROC | Area under ROC | Ranking quality |
| MAE | mean(|y-ŷ|) | Regression, outlier-robust |
| RMSE | sqrt(mean((y-ŷ)²)) | Regression, penalises big errors |
| R² | 1-(SS_res/SS_tot) | Variance explained |

---
## Challenge Exercises

1. **Threshold tuning**: Change `predict_proba` threshold from 0.5 to 0.3. How does precision vs recall change?
2. **Multi-class metrics**: Load `load_iris()` and print a classification report. Note `macro` vs `weighted` average.
3. **RandomizedSearchCV**: Use `RandomizedSearchCV` with `n_iter=20` — it's faster and often just as good.

---
**Next phase:** [Lesson 4.1 — The Artificial Neuron](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-4.1-neuron.ipynb)